# 16 · PCA from a tensors perspective / PCA desde la perspectiva de los tensores

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/project-delphi/tensors-workshop/blob/main/notebooks/16-pca-from-tensors.ipynb)

<div style="height:3px;border-radius:2px;margin:1.4em 0 1.6em;background:linear-gradient(90deg,#0f766e,rgba(15,118,110,0))"></div>

<span style="font:700 11px/1.6 ui-monospace,SFMono-Regular,Menlo,Consolas,monospace;letter-spacing:.18em;color:#0f766e">DEEP DIVE · TAKE-HOME · ABOUT 90 MINUTES / ESTUDIO A FONDO · PARA DESPUÉS · UNOS 90 MINUTOS</span>

Plato's prisoners see shadows. PCA asks which projection preserves the most variation — and leaves open whether that shadow preserves the information a task needs.

Follow one real satellite dataset as a sample-by-feature matrix and as a sample-by-row-by-column-by-band tensor. Derive the matrix views, measure downstream classification and clustering, and explore two meanings of tensor PCA.

<div style="border-left:4px solid rgba(130,130,150,.5);background:rgba(130,130,150,.09);border-radius:0 8px 8px 0;padding:12px 16px;margin:1.2em 0 1.8em;font:400 14.5px/1.7 ui-sans-serif,system-ui,-apple-system,'Segoe UI',Roboto,sans-serif"><div style="font:700 10.5px/1 ui-monospace,SFMono-Regular,Menlo,Consolas,monospace;letter-spacing:.18em;opacity:.62;margin-bottom:10px">🇪🇸 ESPAÑOL</div><div style="margin:0 0 .7em">Los prisioneros de Platón ven sombras. PCA busca la proyección que conserva más variación; queda por comprobar si conserva la información que necesita una tarea.</div><div style="margin:0 0 0">Seguimos datos satelitales reales como matriz de muestras por características y como tensor de muestras, filas, columnas y bandas. Derivamos las vistas matriciales, medimos clasificación y agrupamiento y exploramos dos significados de PCA tensorial.</div></div>

## What you will be able to do / Lo que podrás hacer

- Derive and numerically verify the covariance, eigenvector, SVD and reconstruction identities.
- Explain how correlation PCA changes feature scales and the meaning of distances.
- Evaluate reduced features for classification and clustering using training-only preprocessing.
- Fit a shared multilinear subspace while preserving the sample axis.
- Distinguish multilinear PCA from spiked-tensor recovery and explain their limitations.

<div style="border-left:4px solid rgba(130,130,150,.5);background:rgba(130,130,150,.09);border-radius:0 8px 8px 0;padding:12px 16px;margin:1.2em 0 1.8em;font:400 14.5px/1.7 ui-sans-serif,system-ui,-apple-system,'Segoe UI',Roboto,sans-serif"><div style="font:700 10.5px/1 ui-monospace,SFMono-Regular,Menlo,Consolas,monospace;letter-spacing:.18em;opacity:.62;margin-bottom:10px">🇪🇸 ESPAÑOL</div><ul style="margin:0;padding-left:1.2em"><li style="margin:.35em 0">Derivar y verificar las identidades de covarianza, autovectores, SVD y reconstrucción.</li><li style="margin:.35em 0">Explicar cómo PCA de correlación cambia las escalas y las distancias.</li><li style="margin:.35em 0">Evaluar características reducidas para clasificación y agrupamiento con preprocesamiento ajustado solo en entrenamiento.</li><li style="margin:.35em 0">Ajustar un subespacio multilineal compartido conservando el eje de muestras.</li><li style="margin:.35em 0">Distinguir PCA multilineal de recuperación de una señal tensorial y explicar sus límites.</li></ul></div>

<!-- CORE-PATH -->
## Core path / Ruta esencial

**Take-home deep dive · about 90 minutes.** Work top to bottom; no earlier notebook is required. Recall matrix multiplication and the meaning of a tensor axis. Sections 06, 09 and 10 are useful preparation.

- **Matrix path (35 min):** the cave, satellite patches, covariance, correlation, eigendecomposition, SVD, and a reconstruction exercise.
- **Applications (25 min):** classification and clustering with training-only preprocessing.
- **Tensor path (30 min):** multilinear PCA and a separate spiked-tensor experiment.

Finish with the written **Core activity** at the end. All visible examples run in reading order; folded exercise solutions are optional. The automated runner executes the whole notebook, including solutions.

**Two outcomes:** justify a PCA representation for a measured downstream task; explain which tensor axes a multilinear model compresses and what it assumes.

🇪🇸 Sigue el cuaderno de arriba abajo. Al terminar podrás justificar una representación para una tarea medida y explicar qué ejes comprime el modelo tensorial.

## 1 · Plato’s prisoners: which shadow? / Los prisioneros de Platón

In Book VII of *The Republic*, prisoners see shadows on a wall and take them for the objects themselves. Imagine giving the prisoners a movable wall: which viewing direction would reveal the most variation?

PCA chooses an **orthogonal projection** that keeps as much squared variation as possible. This is our mathematical adaptation: Plato's fire casts perspective shadows, and his allegory concerns knowledge, not an algorithm. The analogy ends before PCA becomes a claim about truth.

**Predict first:** two objects can cast the same shadow. If the wall preserves 99% of the variation, must it preserve their identities? Write your answer before running the counterexample below.

🇪🇸 Una sombra puede conservar mucha variación y perder la identidad. La proyección ortogonal es nuestra adaptación matemática de la alegoría.

![A rotating orthogonal shadow of a synthetic cloud; a cave sketch shows objects, a wall, and seated prisoners.](https://project-delphi.github.io/tensors-workshop/images/cube-16-cave.gif)

## 2 · Setup and real satellite patches / Preparación y parches satelitales

Run this cell once in Colab or in the workshop's `notebooks` environment. It downloads a roughly 101 KB archive and verifies its SHA-256 fingerprint; reruns use a local cache. No GPU or extra pip install is needed.

**Data:** [UCI Statlog Landsat Satellite](https://archive.ics.uci.edu/dataset/146/statlog+landsat+satellite), donated by Ashwin Srinivasan in 1993, CC BY 4.0. It contains 6,435 neighborhoods, with 4,435 supplied training rows and 2,000 supplied test rows. Each label describes the central pixel: six soil/crop classes (codes 1–5 and 7; code 6 is absent).

**Why this application?** Land-cover mapping connects environmental monitoring and agriculture with multispectral machine learning. These are historical sensor values, not current climate observations. We compare representations; we do not estimate present land-cover change.

**Axes:** `patch × row × column × band = N × 3 × 3 × 4`. Bands are approximately green, red and two near-infrared channels. The file lists four bands per pixel, with pixels in row-major order. These are digital sensor values, not calibrated surface reflectance.

**Evaluation limit:** patches come from one small scene and may overlap. Random folds and the supplied split cannot establish geographic or temporal generalization. Coordinates are absent, so a spatial holdout cannot be constructed here. A deployment study needs separate scenes/dates.

🇪🇸 Usamos un conjunto histórico con seis clases. Los parches de una sola escena pueden solaparse; esta evaluación no demuestra generalización a otra región o fecha.

In [ ]:
%matplotlib inline
from pathlib import Path
import hashlib
import io
import itertools
import urllib.request
import zipfile

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import ipywidgets as widgets
from IPython.display import display, Image
from sklearn.base import BaseEstimator, TransformerMixin, clone
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import GridSearchCV, StratifiedKFold
from sklearn.metrics import (accuracy_score, balanced_accuracy_score,
                             adjusted_rand_score, silhouette_score, ConfusionMatrixDisplay)
from sklearn.cluster import KMeans

SEED = 16
URL = "https://archive.ics.uci.edu/static/public/146/statlog+landsat+satellite.zip"
SHA256 = "7c54e0e11c872a1b0b647da370d596dcb06746159cce4121d92ccd70b7d7ce3c"
cache = Path("pca_data") / "landsat.zip"
cache.parent.mkdir(exist_ok=True)
if cache.exists():
    payload = cache.read_bytes()
else:
    try:
        with urllib.request.urlopen(URL, timeout=60) as response:
            payload = response.read()
    except OSError as exc:
        raise RuntimeError("UCI download failed. Check the connection and rerun. / Comprueba la conexión y reintenta.") from exc
if hashlib.sha256(payload).hexdigest() != SHA256:
    raise ValueError("Archive fingerprint changed; inspect the source before using it. / Cambió la huella del archivo.")
cache.write_bytes(payload)
with zipfile.ZipFile(io.BytesIO(payload)) as archive:
    train = np.loadtxt(io.BytesIO(archive.read("sat.trn")))
    test = np.loadtxt(io.BytesIO(archive.read("sat.tst")))
X_train, y_train = train[:, :-1], train[:, -1].astype(int)
X_test, y_test = test[:, :-1], test[:, -1].astype(int)
T_train = X_train.reshape(-1, 3, 3, 4)
T_test = X_test.reshape(-1, 3, 3, 4)
class_names = {1: "red soil", 2: "cotton crop", 3: "grey soil", 4: "damp grey soil",
               5: "vegetation stubble", 7: "very damp grey soil"}
assert X_train.shape == (4435, 36) and X_test.shape == (2000, 36)
assert np.isfinite(train).all() and np.isfinite(test).all()
assert set(y_train) == set(y_test) == set(class_names)
assert np.array_equal(T_train[:, 1, 1, :], X_train[:, 16:20])
print("Training tensor / Tensor de entrenamiento:", T_train.shape)
print("Test tensor / Tensor de prueba:", T_test.shape)
display(pd.DataFrame({"class": class_names.values(),
                      "train": [(y_train == c).sum() for c in class_names],
                      "test": [(y_test == c).sum() for c in class_names]}))

In [ ]:
fig, axes = plt.subplots(2, 4, figsize=(10, 5), constrained_layout=True)
for row, label in enumerate([2, 7]):
    i = np.flatnonzero(y_train == label)[0]
    for b, ax in enumerate(axes[row]):
        ax.imshow(T_train[i, :, :, b], vmin=0, vmax=255, cmap="viridis")
        ax.set_title(f"{class_names[label]}\nband {b+1}", fontsize=10)
        ax.set_xticks([0, 1, 2]); ax.set_yticks([0, 1, 2])
fig.suptitle("Real 3 × 3 patches · shared 0–255 sensor scale (false colour)")
plt.show()

## 3 · One centered matrix, several views / Una matriz centrada, varias vistas

Flattening preserves all 36 values and their ordering. **It is the subsequent model, not reshape, that can ignore the spatial structure.** Let $X$ have shape $(n,p)$, let $\mu$ be its column mean, and set $A=X-\mathbf{1}\mu^T$.

| View | Object and question | Result |
|---|---|---|
| Projection | Which orthonormal $W\in\mathbb R^{p\times k}$ keeps the most variation? | Maximize $\|AW\|_F^2$ with $W^TW=I$ |
| Reconstruction | Which $k$-dimensional affine plane fits best? | Minimize $\|A-AWW^T\|_F^2$ |
| Covariance | Which features vary together, in their chosen units? | $C=A^TA/(n-1)$, shape $(p,p)$ |
| Eigendecomposition | Which axes diagonalize $C$? | $Cv_j=\lambda_jv_j$; sort largest first |
| SVD | Which rank-$k$ matrix approximates $A$? | $A=U\Sigma V^T$; retain $V_k$ |
| Correlation | What if each feature has equal sample variance? | Apply the same machinery to $Z=AD^{-1}$ |

Covariance and correlation are **choices of input geometry**. Eigendecomposition and SVD are **ways to compute** principal axes. They are not four competing names for identical operations.

The bridge is short. A unit direction $w$ has score variance $w^TCw$. Setting the gradient of $w^TCw-\lambda(w^Tw-1)$ to zero gives $Cw=\lambda w$. Taking the largest eigenvalues maximizes retained variance. For any orthonormal $W$,

$$
\|A-AWW^T\|_F^2=\|A\|_F^2-\|AW\|_F^2.
$$

So maximum variance and minimum squared reconstruction error select the same subspace. SVD supplies the identities

$$
C=V\frac{\Sigma^2}{n-1}V^T,\quad S=AV_k=U_k\Sigma_k,\quad \widehat X=SV_k^T+\mu.
$$

Here $V_k$ are **axes** (often called loadings), $S$ are **scores**, and $\lambda_j=\sigma_j^2/(n-1)$ are variances. Some texts use “loadings” for $V_k\sqrt{\Lambda_k}$ instead; check the convention.

Direct SVD avoids explicitly forming $A^TA$, which squares the nonzero singular-value condition ratio. With repeated eigenvalues, signs and bases can differ; compare subspace projectors. A tie across the retained/discarded boundary also makes the optimal subspace nonunique.

🇪🇸 Covarianza y correlación eligen la geometría; eigendescomposición y SVD calculan los ejes. La identidad de proyección conecta varianza y error.

In [ ]:
mu = X_train.mean(axis=0)
A = X_train - mu
n, p = A.shape
C = np.einsum("ni,nj->ij", A, A) / (n - 1)  # contract the sample axis
lam, V_eig = np.linalg.eigh(C)  # symmetric matrix; ascending eigenvalues
order = np.argsort(lam)[::-1]
lam, V_eig = lam[order], V_eig[:, order]
U, singular, Vt = np.linalg.svd(A, full_matrices=False)
k = 12
W = Vt[:k].T
scores = np.einsum("np,pk->nk", A, W)
reconstruction = scores @ W.T
assert np.allclose(C, np.cov(X_train, rowvar=False))
assert np.allclose(lam, singular**2 / (n-1))
assert np.allclose(W @ W.T, V_eig[:, :k] @ V_eig[:, :k].T)
assert np.allclose(reconstruction, (U[:, :k] * singular[:k]) @ Vt[:k])
assert np.allclose(np.cov(scores, rowvar=False), np.diag(lam[:k]))
assert np.isclose(np.sum((A-reconstruction)**2), np.sum(singular[k:]**2))
assert np.isclose(np.sum(A**2), np.sum(scores**2) + np.sum((A-reconstruction)**2))
sk = PCA(n_components=k, svd_solver="full").fit(X_train)
assert np.allclose(sk.components_.T @ sk.components_, W @ W.T)
print("Covariance = eigen = SVD: checks passed / comprobaciones correctas")
print(f"{k} scores from {p} features; variance retained = {lam[:k].sum()/lam.sum():.2%}")
print("Shapes / Formas:", C.shape, W.shape, scores.shape)

## 4 · Covariance or correlation? / ¿Covarianza o correlación?

For $D=\operatorname{diag}(s_1,\ldots,s_p)$, sample standard deviations give $R=Z^TZ/(n-1)=D^{-1}CD^{-1}$. Correlation PCA can give a weak/noisy band as much initial variance as a strong band. Equal variance is a modeling decision, not a universal improvement.

For a correlation-space axis $v$, raw-unit scoring coefficients are $D^{-1}v$, since $Zv=A(D^{-1}v)$. To reconstruct raw values, multiply reconstructed standardized coordinates by $D$ and add $\mu$. Raw-unit scoring and reconstruction vectors therefore differ.

**Predict:** rescale one sensor channel by 100 without changing its information. Which PCA changes? The artificial unit change below is a stress test, not a real sensor calibration.

🇪🇸 Estandarizar cambia las distancias y el peso de las bandas. Un cambio de unidades afecta a PCA de covarianza, pero se cancela en PCA de correlación.

In [ ]:
sd = A.std(axis=0, ddof=1)
assert (sd > 0).all()  # generally: drop constant columns using training data only
Z = A / sd
R = Z.T @ Z / (n - 1)
rz, vz = np.linalg.eigh(R)
uz, sz, vzt = np.linalg.svd(Z, full_matrices=False)
assert np.allclose(R, np.corrcoef(X_train, rowvar=False))
assert np.allclose(rz[::-1], sz**2 / (n-1))
assert np.allclose(vz[:, -k:] @ vz[:, -k:].T, vzt[:k].T @ vzt[:k])
assert np.allclose(Z @ vzt[:k].T, A @ (vzt[:k].T / sd[:, None]))
changed = X_train.copy()
changed[:, 0::4] *= 100  # same band, all nine positions
changed_z = (changed - changed.mean(axis=0)) / changed.std(axis=0, ddof=1)
assert np.allclose(Z, changed_z)
_, _, changed_vt = np.linalg.svd(changed-changed.mean(axis=0), full_matrices=False)
print("Covariance PC1 agreement after unit change:", round(abs(Vt[0] @ changed_vt[0]), 3))
print("Correlation coordinates unchanged / Coordenadas de correlación sin cambios:", np.allclose(Z, changed_z))
fig, axes = plt.subplots(1, 3, figsize=(12, 3.5), constrained_layout=True)
for ax, matrix, title in zip(axes[:2], [C, R], ["Covariance · sensor units²", "Correlation · unitless"]):
    im = ax.imshow(matrix, cmap="coolwarm")
    ax.set(title=title, xlabel="pixel-band feature", ylabel="pixel-band feature")
    fig.colorbar(im, ax=ax, shrink=.75)
axes[2].plot(range(1,37), np.cumsum(lam)/lam.sum(), label="covariance")
axes[2].plot(range(1,37), np.cumsum(sz**2)/np.sum(sz**2), label="correlation")
axes[2].set(xlabel="components", ylabel="cumulative variance share", ylim=(0,1.02))
axes[2].legend(); plt.show()

## 5 · Reconstruct a patch / Reconstruir un parche

A principal axis is itself a $3\times3\times4$ array when reshaped back. An ordinary PCA axis can mix every position and band freely. Move the component slider: see the approximation improve, and see that a reconstruction need not be a valid integer sensor reading.

The GIF follows one real training patch through $k=1,2,4,8,12,24,36$. The slider evaluates a held-out patch using only training axes. The error displayed is for that patch; the optimality theorem concerns the total **training** error.

🇪🇸 Cada eje PCA puede verse como un parche. Reconstruimos una muestra de prueba con ejes aprendidos solo del entrenamiento.

![Four spectral bands of a real Landsat patch reconstructed with increasing PCA rank, alongside the original.](https://project-delphi.github.io/tensors-workshop/images/cube-16-reconstruction.gif)

In [ ]:
def show_patch(k=6, sample=0):
    centered = X_test[sample] - mu
    approx = (centered @ Vt[:k].T) @ Vt[:k] + mu
    fig, axes = plt.subplots(2, 4, figsize=(9, 4), constrained_layout=True)
    for b in range(4):
        axes[0,b].imshow(T_test[sample,:,:,b], vmin=0, vmax=255, cmap="viridis")
        axes[1,b].imshow(approx.reshape(3,3,4)[:,:,b], vmin=0, vmax=255, cmap="viridis")
        axes[0,b].set_title(f"band {b+1}")
        for row in range(2): axes[row,b].set_xticks([]); axes[row,b].set_yticks([])
    axes[0,0].set_ylabel("original"); axes[1,0].set_ylabel(f"k={k}")
    fig.suptitle(f"Held-out patch {sample} · RMSE {np.sqrt(np.mean((X_test[sample]-approx)**2)):.2f} sensor units")
    plt.show()
widgets.interact(show_patch, k=widgets.IntSlider(value=6,min=1,max=36,continuous_update=False),
                 sample=widgets.IntSlider(value=0,min=0,max=49,continuous_update=False));

### Try it: account for the lost energy / Inténtalo: calcula la energía perdida

**Predict → Run → Explain → Check / Predice → Ejecuta → Explica → Comprueba**

Choose $k=4$. Compute the training reconstruction in raw sensor units and its total squared residual. Explain why adding the mean back matters. Compare your residual with the sum of the discarded squared singular values. Try before opening the solution.

🇪🇸 Reconstruye con cuatro componentes, añade la media y compara el error con la energía singular descartada.

In [ ]:
# TODO / TAREA: choose W4, compute Xhat4, and compare the two squared errors.
# Use X_train, mu, Vt, and singular from the visible example.

In [ ]:
#@title Solution / Solución — try first / inténtalo primero { display-mode: 'form' }
W4 = Vt[:4].T
Xhat4 = ((X_train - mu) @ W4) @ W4.T + mu
residual4 = np.sum((X_train - Xhat4)**2)
assert np.isclose(residual4, np.sum(singular[4:]**2))
print("Squared error / Error cuadrático:", round(residual4, 2))

## 6 · The prisoners can still confuse the objects / Aún pueden confundir los objetos

Here is a **synthetic counterexample**, separate from the satellite data. A wide horizontal coordinate is irrelevant to class; a small vertical coordinate carries the entire label. PCA keeps the horizontal shadow.

🇪🇸 En este ejemplo sintético, el eje de poca varianza contiene toda la etiqueta. Conservar varianza no garantiza conservar información para clasificar.

In [ ]:
# --- counterexample / contraejemplo
import numpy as np
x = np.linspace(-30, 30, 100)
cloud = np.column_stack([np.repeat(x, 2), np.tile([-1., 1.], len(x))])
centered = cloud - cloud.mean(axis=0)
_, singular_values, axes = np.linalg.svd(centered, full_matrices=False)
shadow = centered @ axes[:1].T
assert singular_values[0]**2 / np.sum(singular_values**2) > .99
assert np.allclose(shadow[::2], shadow[1::2])
assert np.all(cloud[::2, 1] != cloud[1::2, 1])
# --- end counterexample
labels = (cloud[:, 1] > 0).astype(int)
print(f"Variance kept: {singular_values[0]**2 / np.sum(singular_values**2):.2%}; each shadow has both labels.")
print("The vertical coordinate separates all labels; the shadow cannot.")
fig, ax = plt.subplots(figsize=(8, 2.5))
ax.scatter(cloud[:,0], cloud[:,1], c=labels, cmap="coolwarm", s=14)
ax.set(xlabel="large nuisance coordinate", ylabel="class signal", title="99%+ variance can lose the label")
plt.show()

## 7 · Dimensionality reduction for classification / Reducción para clasificar

Predict the central pixel's land-cover class with logistic regression. Compare all 36 standardized features against PCA with 2, 6, 12 or 24 scores. A two-dimensional visualization is not automatically a useful classifier representation.

We choose component count and regularization `C` with **three stratified training folds**. Each `Pipeline` refits its mean, scale and PCA inside each fold. The supplied test set is evaluated only after selection. Covariance PCA retains sensor units; correlation PCA standardizes before PCA. Scaling scores *after* PCA only conditions the classifier; it does not change the fitted axes. `StandardScaler` uses population variance (`ddof=0`), so its correlation coordinates differ from our sample-standardized ones by a common scalar: axes and variance ratios agree.

**Measure:** balanced accuracy averages recall over classes; ordinary accuracy weights classes by their frequency. Inspect both. There is no promise that PCA improves either. These folds share a scene, so their spread is descriptive, not geographic uncertainty.

🇪🇸 Cada pliegue ajusta su propio preprocesamiento. Elegimos componentes y regularización con entrenamiento; la prueba se reserva para la evaluación final.

In [ ]:
cv = StratifiedKFold(n_splits=3, shuffle=True, random_state=SEED)
base = Pipeline([("scale", StandardScaler()),
                 ("clf", LogisticRegression(max_iter=2000))])
cov_pipe = Pipeline([("pca", PCA(svd_solver="full")),
                     ("score_scale", StandardScaler()),
                     ("clf", LogisticRegression(max_iter=2000))])
corr_pipe = Pipeline([("scale", StandardScaler()),
                      ("pca", PCA(svd_solver="full")),
                      ("score_scale", StandardScaler()),
                      ("clf", LogisticRegression(max_iter=2000))])
searches = {}
for name, pipe in [("All features", base), ("Covariance PCA", cov_pipe), ("Correlation PCA", corr_pipe)]:
    grid = {"clf__C": [0.1, 1., 10.]}
    if "pca" in pipe.named_steps: grid["pca__n_components"] = [2, 6, 12, 24]
    search = GridSearchCV(pipe, grid, scoring="balanced_accuracy", cv=cv, n_jobs=1, error_score="raise")
    search.fit(X_train, y_train)
    searches[name] = search
    print(name, search.best_params_, f"CV balanced accuracy={search.best_score_:.3f}")
fig, ax = plt.subplots(figsize=(7,3.5))
for name in ["Covariance PCA", "Correlation PCA"]:
    results = pd.DataFrame(searches[name].cv_results_)
    best_by_k = results.groupby("param_pca__n_components")["mean_test_score"].max()
    ax.plot(best_by_k.index, best_by_k.values, "o-", label=name)
ax.axhline(searches["All features"].best_score_, color="black", ls="--", label="all 36 features")
ax.set(xlabel="retained scores", ylabel="training CV balanced accuracy", title="Select with training folds")
ax.legend(); plt.show()

## 8 · Tensor PCA: keep the axes meaningful / PCA tensorial: conservar los ejes

“Tensor PCA” can mean different problems. First we learn a shared multilinear subspace for many image tensors. Later we study a single noisy spiked tensor; those objectives are different.

For centered patches $\mathcal A_{nhwb}$, choose orthonormal factors $H\in\mathbb R^{3\times r_h}$, $W\in\mathbb R^{3\times r_w}$ and $B\in\mathbb R^{4\times r_b}$. Scores are a small tensor per sample:

$$
\mathcal G_{nijk}=\sum_{h,w,b}\mathcal A_{nhwb}H_{hi}W_{wj}B_{bk}.
$$

The `n` axis survives: these are **features for each sample**, not a compression of sample identities. The flattened projection basis is $H\otimes W\otimes B$ under our row-major ordering. Unlike ordinary PCA, it has a separable structure. With ranks $(2,2,3)$, there are 12 scores and 24 stored factor entries, versus 432 entries for a dense $36\times12$ basis. Both also store the 36-entry mean; these counts are stored numbers, not independent degrees of freedom.

**Fitting:** initialize with leading left singular vectors of the three feature-mode unfoldings (**HOSVD**). Then update one factor at a time while holding the other two fixed, maximizing total score energy. This alternating refinement implements the multilinear PCA variance objective (also a Tucker/HOOI-style fit with the sample mode uncompressed). Each update solves a matrix SVD; the joint problem is nonconvex. HOSVD alone need not maximize retained energy. Refinement seeks a stationary candidate; a finite iteration budget guarantees neither convergence nor a global optimum.

The reconstruction is an orthogonal projection onto the product subspace. Its flattened scores need not be mutually uncorrelated. “Multilinear” describes the factorization: with fitted factors fixed, transforming a new patch is still linear.

🇪🇸 Comprimimos filas, columnas y bandas; conservamos muestras. HOSVD inicializa y las actualizaciones alternas mejoran la energía, sin garantía de óptimo global.

![The sample axis stays intact while row, column and spectral axes contract from 3×3×4 to 2×2×3.](https://project-delphi.github.io/tensors-workshop/images/cube-16-modes.gif)

In [ ]:
def mode_product(tensor, matrix, axis):
    # matrix[new, old]; put the new axis back where the old one was.
    return np.moveaxis(np.tensordot(matrix, tensor, axes=(1, axis)), 0, axis)

class MultilinearPCA(TransformerMixin, BaseEstimator):
    def __init__(self, ranks=(2, 2, 3), max_iter=15, tol=1e-8):
        self.ranks = ranks
        self.max_iter = max_iter
        self.tol = tol

    def fit(self, X, y=None):
        patches = np.asarray(X).reshape(-1, 3, 3, 4)
        self.mean_ = patches.mean(axis=0)
        centered = patches - self.mean_
        self.factors_ = []
        for axis, rank in enumerate(self.ranks, start=1):
            if not 1 <= rank <= centered.shape[axis]: raise ValueError("Invalid mode rank")
            unfolding = np.moveaxis(centered, axis, 0).reshape(centered.shape[axis], -1)
            u, _, _ = np.linalg.svd(unfolding, full_matrices=False)
            self.factors_.append(u[:, :rank])
        self.energy_ = [np.sum(self._project(centered)**2)]
        for _ in range(self.max_iter):
            for mode, rank in enumerate(self.ranks):
                other_projected = centered
                for j, factor in enumerate(self.factors_):
                    if j != mode: other_projected = mode_product(other_projected, factor.T, j+1)
                unfolding = np.moveaxis(other_projected, mode+1, 0).reshape(centered.shape[mode+1], -1)
                u, _, _ = np.linalg.svd(unfolding, full_matrices=False)
                self.factors_[mode] = u[:, :rank]
            energy = np.sum(self._project(centered)**2)
            self.energy_.append(energy)
            if abs(energy-self.energy_[-2]) <= self.tol * max(energy, 1.): break
        return self

    def _project(self, centered):
        H, W, B = self.factors_
        return np.einsum("nhwb,hi,wj,bk->nijk", centered, H, W, B, optimize=True)

    def transform(self, X):
        centered = np.asarray(X).reshape(-1,3,3,4) - self.mean_
        return self._project(centered).reshape(len(centered), -1)

    def inverse_transform(self, scores):
        H, W, B = self.factors_
        cores = np.asarray(scores).reshape(-1, *self.ranks)
        return (np.einsum("nijk,hi,wj,bk->nhwb", cores, H, W, B, optimize=True)
                + self.mean_).reshape(-1,36)

mpca = MultilinearPCA().fit(X_train)
G = mpca.transform(X_train)
H, W_mode, B = mpca.factors_
product_basis = np.kron(np.kron(H, W_mode), B)
assert G.shape == (4435,12)
assert np.allclose(product_basis.T @ product_basis, np.eye(12))
assert np.allclose(G, A @ product_basis)
assert np.allclose(mpca.inverse_transform(G), G @ product_basis.T + mu)
assert np.all(np.diff(mpca.energy_) >= -1e-7 * np.sum(A**2))
mpca_error = np.sum((X_train-mpca.inverse_transform(G))**2)
assert np.isclose(mpca_error + np.sum(G**2), np.sum(A**2))
assert mpca_error >= np.sum(singular[12:]**2) - 1e-6  # unconstrained PCA is optimal at rank 12
full_tensor = MultilinearPCA(ranks=(3,3,4)).fit(X_train)
assert np.allclose(full_tensor.inverse_transform(full_tensor.transform(X_test)), X_test)
print("Multilinear scores / Puntuaciones multilineales:", G.shape)
print("Stored basis entries: tensor", sum(f.size for f in mpca.factors_), "dense PCA", 36*12)
print("HOSVD → refined retained energy:", np.round(np.array([mpca.energy_[0],mpca.energy_[-1]]) / np.sum(A**2), 5))
print("PCA-12 / multilinear relative training errors:",
      np.sum(singular[12:]**2)/np.sum(A**2), mpca_error/np.sum(A**2))

### Same score budget, same held-out task / Mismo presupuesto y tarea

Compare dense covariance PCA-12 with multilinear `(2,2,3)` features. Ranks are fixed in advance for this matched budget; only classifier regularization is selected by training CV. A production study could tune mode ranks inside those folds too. The all-feature and selected PCA routes remain in the table.

Training reconstruction error must favor unrestricted PCA at equal output dimension. Held-out classification may disagree because variance retention and class separation are different objectives.

🇪🇸 A igual dimensión, PCA gana en error de entrenamiento; la clasificación puede dar otro orden. Los rangos tensoriales se fijan antes de mirar la prueba.

In [ ]:
for name, reducer in [("PCA-12 matched", PCA(n_components=12, svd_solver="full")),
                      ("Multilinear-12", MultilinearPCA())]:
    pipe = Pipeline([("reduce", reducer), ("score_scale", StandardScaler()),
                     ("clf", LogisticRegression(max_iter=2000))])
    searches[name] = GridSearchCV(pipe, {"clf__C": [.1, 1., 10.]}, scoring="balanced_accuracy",
                                  cv=cv, n_jobs=1, error_score="raise").fit(X_train, y_train)
rows = []
for name, search in searches.items():
    predicted = search.predict(X_test)
    steps = search.best_estimator_.named_steps
    dim = (steps["pca"].n_components_ if "pca" in steps else
           12 if "reduce" in steps else 36)
    rows.append({"representation":name, "features":dim, "CV balanced accuracy":search.best_score_,
                 "test accuracy":accuracy_score(y_test,predicted),
                 "test balanced accuracy":balanced_accuracy_score(y_test,predicted)})
classification_results = pd.DataFrame(rows).set_index("representation")
display(classification_results.round(3))
# Select which confusion matrix to inspect using CV, not the test table.
chosen_name = max(searches, key=lambda name: searches[name].best_score_)
fig, ax = plt.subplots(figsize=(7,6))
ConfusionMatrixDisplay.from_predictions(y_test, searches[chosen_name].predict(X_test),
    labels=list(class_names), display_labels=list(class_names.values()),
    normalize="true", xticks_rotation=45, ax=ax, colorbar=False, values_format=".2f")
ax.set_title(f"CV-selected: {chosen_name} · test recall by class")
plt.tight_layout(); plt.show()

### Read the evidence / Leer la evidencia

A reference run selects 24 components for both PCA routes. Held-out balanced accuracy is approximately 0.794 for all 36 features, 0.806 for covariance PCA-24 and 0.798 for correlation PCA-24. At the matched twelve-score budget, dense PCA gives about 0.788 and multilinear PCA about 0.774. Small differences can depend on solver versions and sampling; this is one benchmark comparison, not a universal ranking or significance test.

The tensor basis stores 18 times fewer entries (24 versus 432), but classifier coefficients and the mean still take space. Inspect the confusion matrix: damp grey soil has much lower recall than cotton or red soil. A compact representation can preserve most variation while leaving a mapping-relevant distinction unresolved.

🇪🇸 El ejemplo favorece ligeramente PCA de covarianza con 24 componentes; la representación tensorial ahorra almacenamiento con menor precisión. Examina los errores por clase antes de recomendar un mapa.

## 9 · PCA before clustering / PCA antes de agrupar

K-means sees distances, not class labels. Fit three representations on training features: standardized originals, 2-PC coordinates and 12-PC coordinates. We use six clusters as a declared benchmark choice because there are six known classes; this is not discovering the number of land-cover types. `n_init=10` tries multiple initializations.

Cluster the **training** representation, then assign held-out patches to the learned centers. Labels enter only afterward through **adjusted Rand index (ARI)**, a permutation-invariant agreement score: 1 is exact agreement, roughly 0 is chance-level agreement under its reference model, and negative values are possible. Clusters need not correspond to soil labels.

Silhouette measures separation by distances without labels. We report it in one **common standardized original feature space** so representation changes cannot silently change the measuring stick. The two-PC plot is a view of the assignments; apparent separation there does not establish separation in every dimension.

🇪🇸 K-means no ve etiquetas. ARI compara grupos y clases después; la silueta se calcula en el mismo espacio original estandarizado para todas las rutas.

In [ ]:
cluster_scaler = StandardScaler().fit(X_train)
cluster_train = cluster_scaler.transform(X_train)
cluster_test = cluster_scaler.transform(X_test)
cluster_rows, cluster_labels = [], {}
plot_pca = PCA(n_components=2, svd_solver="full").fit(cluster_train)
plot_test = plot_pca.transform(cluster_test)
for dimension in [36, 2, 12]:
    if dimension == 36:
        train_rep, test_rep = cluster_train, cluster_test
    else:
        reducer = PCA(n_components=dimension, svd_solver="full").fit(cluster_train)
        train_rep, test_rep = reducer.transform(cluster_train), reducer.transform(cluster_test)
    km = KMeans(n_clusters=6, n_init=10, random_state=SEED).fit(train_rep)
    assigned = km.predict(test_rep)
    cluster_labels[dimension] = assigned
    silhouette = (silhouette_score(cluster_test, assigned, sample_size=1000, random_state=SEED)
                  if 1 < len(np.unique(assigned)) < len(assigned) else np.nan)
    cluster_rows.append({"features":dimension, "test ARI":adjusted_rand_score(y_test, assigned),
                         "test silhouette (common space)":silhouette})
display(pd.DataFrame(cluster_rows).set_index("features").round(3))
fig, axes = plt.subplots(1,2,figsize=(10,4),constrained_layout=True)
for ax, colors, title in [(axes[0], y_test, "Known classes (evaluation only)"),
                          (axes[1], cluster_labels[12], "K-means in 12 PCs")]:
    for label in np.unique(colors):
        mask = colors == label
        ax.scatter(plot_test[mask,0],plot_test[mask,1],s=5,alpha=.35,label=str(label))
    ax.set(title=title,xlabel="PC1",ylabel="PC2"); ax.legend(markerscale=2,fontsize=8)
plt.show()

### What did compression change? / ¿Qué cambió la compresión?

The reference ARI values are about 0.522 for all features, 0.508 for two PCs and 0.518 for twelve PCs. In the common original space, silhouettes are about 0.350, 0.346 and 0.351. Here, a dramatic reduction in dimension produces similar clustering scores; PCA does not make the clusters coincide with the known classes. Do not treat the tiny silhouette difference as evidence of a reliable improvement.

🇪🇸 Reducir la dimensión conserva resultados parecidos de agrupamiento, sin convertir grupos en clases ni demostrar una mejora fiable.

## 10 · A different tensor PCA: recover a hidden direction / Recuperar una dirección oculta

Now leave the satellite benchmark. In a **synthetic spiked model**, observe one symmetric order-3 tensor

$$
\mathcal Y=\beta\,v\otimes v\otimes v+\mathcal E,\qquad \|v\|_2=1.
$$

The task is to recover the planted direction $v$ from noise. A rank-one fit maximizes the absolute contraction $|\sum_{ijk}Y_{ijk}u_i u_j u_k|$ on the unit sphere. For fixed $u$, that contraction is the optimal signed amplitude. Compare with the quadratic matrix objective $u^TCu$: the tensor objective is cubic, and a power-style step contracts **two** indices,

$$
u_{t+1}\propto\mathcal Y(I,u_t,u_t).
$$

Our symmetric noise is the average of all six permutations of an i.i.d. Gaussian tensor, scaled by $1/\sqrt d$. Its entries are correlated after symmetrization. This explicitly defined toy convention is **not** a numerical reproduction of a paper's recovery thresholds.

We try several initial directions, keep the visited iterate with largest absolute objective, and evaluate $|u^Tv|$ only because simulation reveals the truth. Tensor power iteration can cycle or find poor candidates; keeping the best visited point does not certify global optimality. The rank-one problem has no matrix-like eigenvalue ordering that solves general higher-order cases. Statistical recoverability and an algorithm's ability to find a signal are separate questions.

🇪🇸 Este experimento es sintético y distinto de MPCA. Contraemos dos índices para buscar una señal plantada; los reinicios no certifican un óptimo global.

In [ ]:
rng = np.random.default_rng(SEED)
d = 12
planted = rng.normal(size=d); planted /= np.linalg.norm(planted)
raw_noise = rng.normal(size=(d,d,d)) / np.sqrt(d)
noise = sum(raw_noise.transpose(perm) for perm in itertools.permutations(range(3))) / 6
spike = np.einsum("i,j,k->ijk", planted, planted, planted)
starts = rng.normal(size=(16,d)); starts /= np.linalg.norm(starts,axis=1,keepdims=True)

def tensor_power_candidates(Y, initial, steps=80):
    records = []
    for start in initial:
        u = start.copy()
        best_u, best_value = u.copy(), abs(np.einsum("ijk,i,j,k",Y,u,u,u))
        for _ in range(steps):
            update = np.einsum("ijk,j,k->i",Y,u,u)
            norm = np.linalg.norm(update)
            if norm < 1e-14: break
            u = update/norm
            value = abs(np.einsum("ijk,i,j,k",Y,u,u,u))
            if value > best_value: best_u, best_value = u.copy(), value
        records.append((best_value,best_u))
    return records

spiked_rows=[]
for beta in [0., .5, 1., 2., 4., 8.]:
    Y = beta*spike + noise
    candidates = tensor_power_candidates(Y, starts)
    objective, estimated = max(candidates,key=lambda item:item[0])  # no access to planted here
    alignments = [abs(vector @ planted) for _,vector in candidates]
    spiked_rows.append({"signal amplitude":beta,"chosen alignment":abs(estimated @ planted),
                        "worst restart alignment":min(alignments),"objective":objective})
# Meaningful noiseless check: contraction recovers v from a nonorthogonal start.
_, noiseless = max(tensor_power_candidates(3*spike,starts),key=lambda item:item[0])
assert np.isclose(abs(noiseless @ planted),1.)
spiked_results=pd.DataFrame(spiked_rows)
display(spiked_results.round(3))
fig,ax=plt.subplots(figsize=(7,3.5))
ax.plot(spiked_results["signal amplitude"],spiked_results["chosen alignment"],"o-",label="chosen by objective")
ax.plot(spiked_results["signal amplitude"],spiked_results["worst restart alignment"],"o--",label="worst restart (diagnostic)")
ax.set(xlabel="signal amplitude β (fixed toy noise)",ylabel="|estimated direction · planted direction|",ylim=(0,1.05))
ax.legend(); plt.show()

### Pause the pictures / Pausa las imágenes

The GIFs are precomputed teaching assets; the calculations above reproduce their ideas. Use this optional frame stepper to inspect any frame. It needs the published site, so on an unpublished branch use the committed files in `images/`. A missing animation does not prevent the data examples from running.

🇪🇸 El selector permite inspeccionar cada fotograma. Usa los archivos locales si la rama aún no está publicada.

In [ ]:
from PIL import Image as PILImage, ImageSequence

def inspect_animation(scene="cave", frame=0):
    name = f"cube-16-{scene}.gif"
    candidates = [Path("images")/name, Path("../images")/name]
    try:
        local = next((path for path in candidates if path.exists()), None)
        if local:
            data = local.read_bytes()
        else:
            with urllib.request.urlopen(f"https://project-delphi.github.io/tensors-workshop/images/{name}",timeout=10) as response:
                data = response.read()
        with PILImage.open(io.BytesIO(data)) as picture:
            index = min(frame, picture.n_frames-1)
            picture.seek(index)
            print(f"Frame / Fotograma {index+1}/{picture.n_frames}")
            display(picture.convert("RGB"))
    except OSError:
        print("Animation unavailable; use images/ in the checkout. / Usa images/ en la copia local.")
widgets.interact(inspect_animation, scene=["cave","reconstruction","modes"],
                 frame=widgets.IntSlider(value=0,min=0,max=35,continuous_update=False));

## Core activity · Choose a representation / Elige una representación

**Predict → Run → Explain → Check / Predice → Ejecuta → Explica → Comprueba**

1. **Predict:** before consulting the results, choose covariance PCA, correlation PCA, all features, or multilinear features for land-cover classification. State your reason.
2. **Run:** execute the examples in order. Record feature count, CV score, held-out balanced accuracy, ARI, and tensor basis storage. Keep classification and clustering claims separate.
3. **Explain:** did your preferred representation survive the comparison? Name the preprocessing and shape of its scores. Explain why a larger variance share need not imply better labels, and why the tensor model is constrained.
4. **Check:** your evidence must use training-fitted transforms, include an all-feature baseline, distinguish the two meanings of tensor PCA, and acknowledge the single-scene evaluation limit. No single winning method is prescribed.

**Transfer:** design a study on a new satellite scene. What will be held out? Which band scales and mode ranks will be chosen inside training? Which class confusions would matter to the mapping task?

**Core complete.** Keep a short evidence-based recommendation. For further exploration, vary the class-independent noise in the shadow example, compare HOSVD initialization with refinement, or repeat the spiked experiment over many independent noise draws before discussing recovery trends.

🇪🇸 Entrega una recomendación breve con métricas, formas y límites. Propón una evaluación en otra escena; distingue compresión, clasificación, agrupamiento y recuperación de señales.

## Sources and next steps / Fuentes y siguientes pasos

- [Six views of PCA](https://project-delphi.github.io/ml-blog/posts/six-views-of-pca/): companion derivation connecting variance, reconstruction, eigenvectors and SVD; this notebook adds the satellite and tensor experiments.
- [UCI Landsat data and documentation](https://archive.ics.uci.edu/dataset/146/statlog+landsat+satellite): Srinivasan (1993), DOI 10.24432/C55887, CC BY 4.0. The downloaded archive contains `sat.doc` with feature ordering and class definitions.
- [scikit-learn PCA API](https://scikit-learn.org/stable/modules/generated/sklearn.decomposition.PCA.html): centering, solver choices, scores and variance conventions.
- [Lu, Plataniotis & Venetsanopoulos (2008), MPCA](https://www.comm.toronto.edu/~kostas/Publications2008/pub/102.pdf): multilinear feature extraction and alternating optimization.
- [Montanari & Richard (2014), A statistical model for tensor PCA](https://arxiv.org/abs/1411.1076): the spiked-tensor problem and computational/statistical questions.
- [Plato, The Republic, Book VII](https://www.gutenberg.org/files/1497/1497-h/1497-h.htm): the cave allegory, in Benjamin Jowett's translation.

Continue with [Tucker decomposition](https://colab.research.google.com/github/project-delphi/tensors-workshop/blob/main/notebooks/10-tucker-decomposition.ipynb) or [CP factorization](https://colab.research.google.com/github/project-delphi/tensors-workshop/blob/main/notebooks/14-cp-factorization.ipynb).

🇪🇸 Las fuentes separan la geometría matricial, MPCA, el modelo de señal plantada y la alegoría original.

<div style="height:3px;border-radius:2px;margin:1.4em 0 1.6em;background:linear-gradient(90deg,#0f766e,rgba(15,118,110,0))"></div>

## Done with this deep dive / Fin de este estudio a fondo

Next deep dive / Siguiente estudio a fondo: **17 · 4D multi-head attention: reshape Q, K and V / Atención multicabeza 4D: reorganiza Q, K y V** — [open in Colab](https://colab.research.google.com/github/project-delphi/tensors-workshop/blob/main/notebooks/17-multi-head-attention.ipynb).

[← Workshop site / Sitio del taller](https://project-delphi.github.io/tensors-workshop/) · [All notebooks / Todos los notebooks](https://project-delphi.github.io/tensors-workshop/notebooks.html) · [Handbook / Manual](https://project-delphi.github.io/tensors-workshop/tensors_workshop_plan_with_quizzes.html)